# 🛰️ ICARUS_WORKER_EO — Earth Observation Worker
## Dedicated Remote Scientific Worker Runtime (Kaggle Account 2)

**Version:** 0.3.0 (Genuine Prithvi-EO-2.0-300M Integration)  
**Role:** Earth Observation Processing & Specialist Foundation Model Host  
**Primary Model:** IBM/NASA Prithvi-EO-2.0-300M (Revision: `9eb1b1102806593963daa333bcc491b1c6f8562f`)  
**Parameters:** 330,419,712  
**Interface:** FastAPI over Encrypted ngrok Tunnel  
**Deployment Profile:** Kaggle Account 2 (GPU T4 x2 or P100)

---

### Worker Responsibilities in 4-Account Topology
- Hosts traditional multispectral algorithms: **NDVI, NDWI, NBR, Area with CRS validation**.
- Provides **GeoTIFF ingestion, band mapping, spatial tiling (224x224), and 4-frame temporal sequence prep**.
- Hosts genuine **IBM/NASA Prithvi-EO-2.0-300M** with lazy loading, dynamic `pos_embed` pruning, official climatological normalization, and VRAM eviction.
- Serves `/health`, `/capabilities`, `/analyze`, and `/unload-model` endpoints to `ICARUS_CORE`.


---
## 1. Environment Detection & Kaggle Secrets

Inspect GPU resources (VRAM availability) and retrieve `NGROK_AUTH_TOKEN` from Kaggle Secrets.


In [ ]:
# ============================================================
# 1. Setup, Environment & Secrets
# ============================================================
import os
import sys
import json
import time
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

def get_secret(key: str, default: str = "") -> str:
    """Fetches secret from Kaggle Secrets or environment variables."""
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            return user_secrets.get_secret(key)
        except Exception:
            pass
    return os.environ.get(key, default)

NGROK_AUTH_TOKEN = get_secret("NGROK_AUTH_TOKEN")
COPERNICUS_CLIENT_ID = get_secret("COPERNICUS_CLIENT_ID")
NASA_EARTHDATA_USER = get_secret("NASA_EARTHDATA_USERNAME")

# Inspect GPU and PyTorch
try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "None"
    VRAM_MB = torch.cuda.get_device_properties(0).total_memory // (1024**2) if CUDA_AVAILABLE else 0
except ImportError:
    CUDA_AVAILABLE = False
    GPU_NAME = "PyTorch not installed"
    VRAM_MB = 0

print("=" * 65)
print("🛰️  ICARUS_WORKER_EO Runtime Initialization")
print("=" * 65)
print(f"  Platform:          {'Kaggle Notebook' if IS_KAGGLE else 'Local / Server'}")
print(f"  Python Version:    {sys.version.split()[0]}")
print(f"  GPU Accelerator:   {GPU_NAME} ({VRAM_MB} MB VRAM)")
print(f"  CUDA Available:    {CUDA_AVAILABLE}")
print(f"  NGROK_AUTH_TOKEN:  {'✅ Configured' if NGROK_AUTH_TOKEN else '⚠️ Missing (Required for public tunnel)'}")
print("=" * 65)

---
## 2. Core Library Imports

Import the shared schemas, EO pipeline, and Prithvi foundation adapter.


In [ ]:
# ============================================================
# 2. Package Imports
# ============================================================
try:
    import icarus
except ImportError:
    # Ensure working directory is in sys.path
    repo_path = Path("/kaggle/working/ICARUS").resolve()
    if repo_path.exists():
        sys.path.insert(0, str(repo_path))
    else:
        sys.path.insert(0, str(Path(".").resolve()))

from icarus.schemas import (
    DomainEnum,
    CapabilityStatus,
    TaskStatus,
    WorkerRuntimeStatus,
    ResearchRequest,
    ResearchResponse,
    TaskMetric,
    ExecutionMetadata,
    CapabilityDescriptor,
    WorkerCapabilitiesResponse,
    HealthCheckResponse,
    UnloadModelRequest,
    UnloadModelResponse,
)
from icarus.workers.eo.eo_pipeline import EOPipeline
from icarus.workers.eo.prithvi_adapter import PrithviEOAdapter, PrithviModelStatus
from icarus.workers.eo.service import app

print("✅ EO Worker modules and FastAPI application loaded.")

---
## 3. Traditional Spectral Pipeline & Spatio-Temporal Sequence Preparation

Verify traditional spectral algorithms (NDVI, NDWI, NBR, surface area with CRS checks, spatial tiling, and 4-frame temporal sequence prep) independently of deep learning models.


In [ ]:
# ============================================================
# 3. Traditional EO Pipeline & Temporal Prep Verification
# ============================================================
import numpy as np

# 1. Test NDVI math with synthetic arrays
nir_demo = np.array([[0.6, 0.7], [0.8, 0.5]])
red_demo = np.array([[0.2, 0.1], [0.2, 0.3]])
ndvi_calc = EOPipeline.compute_ndvi(nir_demo, red_demo)
print("NDVI Sample Output:", ndvi_calc)

# 2. Test vegetation distribution statistics
stats = EOPipeline.compute_vegetation_statistics(ndvi_calc)
print("\nVegetation Statistics Summary:")
print(f"  Mean NDVI: {stats['mean']}")
print(f"  Dense Canopy Fraction: {stats['classification_fractions']['dense_canopy_fraction']}")

# 3. Test CRS Guard on Area Calculation
area_deg = EOPipeline.calculate_surface_area_km2(1000, 10.0, crs="EPSG:4326")
print("\nGeographic Area Calculation Guard:", area_deg.get("error", "Passed"))

area_utm = EOPipeline.calculate_surface_area_km2(10000, 10.0, crs="EPSG:32643")
print(f"Projected Area Calculation: {area_utm['area_km2']} {area_utm['unit']}")

# 4. Test 4-Observation Temporal Sequence Preparation
single_obs = np.random.uniform(500, 3000, (6, 250, 250)).astype(np.float32)
temporal_seq, prov = EOPipeline.prepare_temporal_sequence(
    single_obs,
    repeat_single_frame=True,
    crop_to_224=True,
)
print(f"\nPrepared Temporal Sequence Shape: {temporal_seq.shape} (Channels=6, Time=4, H=224, W=224)")
print(f"Strategy: {prov['temporal_strategy']}")

---
## 4. IBM/NASA Prithvi-EO-2.0-300M Adapter & Smoke Test

Verify official checkpoint loading (`Prithvi_EO_V2_300M.pt`), parameter count verification (330,419,712), dynamic `pos_embed` removal, and forward feature extraction.


In [ ]:
# ============================================================
# 4. Genuine Prithvi-EO-2.0-300M Smoke Test
# ============================================================
from scripts.kaggle_prithvi_smoke_test import run_prithvi_smoke_test

# Run full official verification (downloads checkpoint, tests forward pass, checks finiteness)
if CUDA_AVAILABLE:
    run_prithvi_smoke_test()
else:
    print("⚠️ Running without GPU. You can run the smoke test, or use the lazy-loaded adapter directly.")
    adapter = PrithviEOAdapter(device="cpu")
    print("Prithvi Status:", json.dumps(adapter.get_status(), indent=2))

---
## 5. In-Process HTTP Contract Verification

Run verification requests directly against the FastAPI `app` using `starlette.testclient.TestClient` before starting the public network server.


In [ ]:
# ============================================================
# 5. In-Process Contract Verification
# ============================================================
from starlette.testclient import TestClient

client = TestClient(app)

print("--- [1. GET /health] ---")
resp_health = client.get("/health")
print(f"Status: {resp_health.status_code}")
print(json.dumps(resp_health.json(), indent=2))

print("\n--- [2. GET /capabilities] ---")
resp_caps = client.get("/capabilities")
print(f"Status: {resp_caps.status_code}")
print(f"Total Capabilities Advertised: {len(resp_caps.json()['capabilities'])}")

print("\n--- [3. POST /analyze (NDVI Calculation)] ---")
req_ndvi = {
    "domain": "earth_observation",
    "task": "eo_vegetation_analysis",
    "research_question": "Compute NDVI for study quadrant",
    "parameters": {
        "nir": [[0.7, 0.8], [0.6, 0.75]],
        "red": [[0.15, 0.2], [0.18, 0.22]],
    },
}
resp_analyze = client.post("/analyze", json=req_ndvi)
print(f"Status: {resp_analyze.status_code}")
print(json.dumps(resp_analyze.json(), indent=2))

print("\n--- [4. POST /analyze (Prithvi Demo Mode Rejection vs Execution)] ---")
# Without data and without allow_synthetic_demo -> Expected honest rejection
req_missing = {
    "domain": "earth_observation",
    "task": "eo_foundation_embeddings",
    "research_question": "Extract latent representations",
    "allow_synthetic_demo": False,
}
resp_missing = client.post("/analyze", json=req_missing)
print(f"Honest Rejection Status: {resp_missing.status_code} -> Code: {resp_missing.json()['error']['code']}")

print("\n--- [5. POST /unload-model] ---")
resp_unload = client.post("/unload-model", json={"model_id": "ibm-nasa-geospatial/Prithvi-EO-2.0-300M"})
print(f"Status: {resp_unload.status_code}")
print(json.dumps(resp_unload.json(), indent=2))

---
## 6. Launch Public FastAPI Server via ngrok

Expose the worker service to the internet using pyngrok.  
The public URL printed below can be registered on Kaggle Account 1 (`ICARUS_CORE.ipynb`).


In [ ]:
# ============================================================
# 6. Start Public ngrok Tunnel & Background Server
# ============================================================
import threading
import uvicorn

PORT = 8000

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

# Start Uvicorn in background thread
server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)
print(f"✅ Local FastAPI worker listening on http://localhost:{PORT}")

# Start ngrok tunnel if token is available
if NGROK_AUTH_TOKEN:
    try:
        from pyngrok import ngrok
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        # Open HTTP tunnel
        public_tunnel = ngrok.connect(PORT, "http")
        public_url = public_tunnel.public_url
        print("\n" + "=" * 65)
        print("🌐 ICARUS_WORKER_EO PUBLIC NGROK TUNNEL ONLINE:")
        print(f"   {public_url}")
        print("=" * 65)
        print("\n👉 Copy this URL and register it on Account 1 (ICARUS_CORE):")
        print(f'   orchestrator.register_worker("{public_url}", name="ICARUS_WORKER_EO")')
    except Exception as e:
        print(f"⚠️ ngrok tunnel launch failed: {str(e)}")
else:
    print("\n⚠️ NGROK_AUTH_TOKEN not provided.")
    print("   To connect this worker to ICARUS_CORE in a multi-notebook setup,")
    print("   add NGROK_AUTH_TOKEN to Kaggle Secrets (Add-ons > Secrets).")